# 02 - xBD Preprocessing Visual Checks

Run preprocessing pipeline and visually validate generated masks, crops, and splits.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    RAW_DIR = Path('/kaggle/input/xview2-challenge-dataset-train-and-test')
    OUT_DIR = Path('/kaggle/working/processed_xbd')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q rasterio shapely scikit-image
else:
    RAW_DIR = Path('../data/raw/xbd')
    OUT_DIR = Path('../data/processed/xbd')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Raw dir: {RAW_DIR} (exists: {RAW_DIR.exists()})')
print(f'Output dir: {OUT_DIR}')

## Run Preprocessing Pipeline

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.xbd_preprocess import preprocess_xbd

preprocess_xbd(
    raw_dir=RAW_DIR,
    out_dir=OUT_DIR,
    make_crops=True,
    make_visualizations=True,
    crop_margin=16,
    # debug_limit=50,  # Uncomment for quick test
)

## Verify Output Structure

In [ ]:
# Count output files
for subdir in ['images/pre', 'images/post', 'masks/building_binary', 'masks/damage_multiclass', 'masks/change_binary', 'crops/building_pre', 'crops/building_post', 'metadata']:
    d = OUT_DIR / subdir
    count = len(list(d.glob('*'))) if d.exists() else 0
    print(f'{subdir}: {count} files')

In [ ]:
import pandas as pd

# Load and display metadata
tiles_csv = OUT_DIR / 'metadata' / 'tiles.csv'
buildings_csv = OUT_DIR / 'metadata' / 'buildings.csv'
splits_csv = OUT_DIR / 'metadata' / 'splits_event_aware.csv'

if tiles_csv.exists():
    tiles_df = pd.read_csv(tiles_csv)
    print('Tiles CSV:')
    print(tiles_df.head())
    print(f'\nTotal tiles: {len(tiles_df)}')
    print(f'Split distribution:\n{tiles_df["split"].value_counts()}')

if buildings_csv.exists():
    buildings_df = pd.read_csv(buildings_csv)
    print(f'\nBuildings CSV: {len(buildings_df)} crops')
    print(f'Damage distribution:\n{buildings_df["label"].value_counts()}')

if splits_csv.exists():
    splits_df = pd.read_csv(splits_csv)
    print(f'\nEvent-aware splits:\n{splits_df["split"].value_counts()}')

## Visualize Generated Masks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rescuevision.constants import DAMAGE_COLOR_MAP

# Visualize a few tiles
pre_dir = OUT_DIR / 'images' / 'pre'
post_dir = OUT_DIR / 'images' / 'post'
bldg_dir = OUT_DIR / 'masks' / 'building_binary'
dmg_dir = OUT_DIR / 'masks' / 'damage_multiclass'
chng_dir = OUT_DIR / 'masks' / 'change_binary'

samples = sorted(pre_dir.glob('*.npy'))[:4]

fig, axes = plt.subplots(len(samples), 5, figsize=(25, 5*len(samples)))
if len(samples) == 1:
    axes = axes.reshape(1, -1)

for idx, pre_path in enumerate(samples):
    tile_id = pre_path.stem
    
    pre_img = np.load(pre_path)
    post_img = np.load(post_dir / f'{tile_id}.npy')
    bldg_mask = np.load(bldg_dir / f'{tile_id}.npy')
    dmg_mask = np.load(dmg_dir / f'{tile_id}.npy')
    chng_mask = np.load(chng_dir / f'{tile_id}.npy')
    
    # Colorize damage mask
    dmg_color = np.zeros((*dmg_mask.shape, 3), dtype=np.uint8)
    for cls_id, color in DAMAGE_COLOR_MAP.items():
        dmg_color[dmg_mask == cls_id] = color
    
    axes[idx, 0].imshow(pre_img)
    axes[idx, 0].set_title('Pre-disaster')
    axes[idx, 1].imshow(post_img)
    axes[idx, 1].set_title('Post-disaster')
    axes[idx, 2].imshow(bldg_mask, cmap='gray')
    axes[idx, 2].set_title('Building Mask')
    axes[idx, 3].imshow(dmg_color)
    axes[idx, 3].set_title('Damage Mask')
    axes[idx, 4].imshow(chng_mask, cmap='hot')
    axes[idx, 4].set_title('Change Mask')
    
    for ax in axes[idx]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR / 'mask_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mask visualization saved.')

## Save Processed Data as Kaggle Dataset

In [ ]:
if IS_KAGGLE:
    print('Processed data is at:', OUT_DIR)
    print('Save this notebook output as a Kaggle dataset for training notebooks.')
    print('\nDirectory size:')
    !du -sh {OUT_DIR}
    !ls {OUT_DIR}/metadata/
else:
    print('Running locally. Processed data saved to:', OUT_DIR)